# Day 17 - Advanced Retrieval with Metadata
Upgrade of the Day 15 RAG pipeline: every document now carries `source`, `category`, `date`, `document_type`, and retrieval can be constrained by a filter dictionary.

**Why:** pure vector search is context-blind. It cannot tell a 2021 policy from a 2024 one, or a technical reference from a marketing blurb, because the embeddings look similar.

In [1]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Step 1 - Knowledge base with metadata
Same fictional *Veridia Labs* world as Day 15. Each document is a LangChain `Document` with `page_content` (text) + `metadata` (structure). Old versions and marketing blurbs are included on purpose, so plain vector search has something to get wrong.

In [2]:
from langchain_core.documents import Document

raw = [
 # id, text, source, category, document_type, date
 (0,"Refund policy (2021): Customers may request a refund within 14 days of purchase on any plan.","policy_2021.pdf","pricing","policy","2021-03-01"),
 (1,"Refund policy (2024): Customers may request a refund within 30 days of purchase. Annual plans are refunded pro-rata after 30 days.","policy_2024.pdf","pricing","policy","2024-02-10"),
 (2,"Veridia Labs offers flexible plans with generous refund guarantees so you can try risk-free.","website.html","pricing","marketing","2024-05-01"),
 (3,"Pricing: Starter costs $19 per month, Pro costs $49 per month, Enterprise is custom.","pricing_faq.md","pricing","faq","2023-06-15"),
 (4,"Pricing (2020): Starter costs $9 per month, Pro costs $29 per month.","pricing_2020.md","pricing","policy","2020-01-20"),
 (5,"Veridia Vault encrypts all data at rest with AES-256 and in transit with TLS 1.3.","security_ref_2024.pdf","security","technical_reference","2024-03-12"),
 (6,"Data is encrypted at rest with AES-128 and in transit with TLS 1.2.","security_ref_2020.pdf","security","technical_reference","2020-08-05"),
 (7,"Bank-grade security keeps your data safe. Veridia Labs takes privacy and encryption seriously.","website.html","security","marketing","2024-05-01"),
 (8,"Data retention: deleted customer data is permanently purged from backups within 90 days.","retention_policy.pdf","security","policy","2023-09-01"),
 (9,"Support hours: Monday to Friday, 9am to 6pm IST. Enterprise customers get 24/7 phone support.","support_faq_2024.md","support","faq","2024-01-08"),
 (10,"Support hours: Monday to Friday, 10am to 5pm IST. Email support only.","support_faq_2021.md","support","faq","2021-04-02"),
 (11,"Our friendly support team is always ready to help you succeed with Veridia Labs.","website.html","support","marketing","2024-05-01"),
 (12,"Veridia API rate limit: 1000 requests per minute on Pro, 5000 on Enterprise. Exceeding the limit returns HTTP 429.","api_ref_2024.pdf","product","technical_reference","2024-06-20"),
 (13,"Veridia API rate limit: 100 requests per minute for all plans.","api_ref_2021.pdf","product","technical_reference","2021-07-11"),
 (14,"Veridia Labs APIs are fast, scalable and developer friendly.","website.html","product","marketing","2024-05-01"),
 (15,"Release 4.2 (2025): added single sign-on via SAML and a dark mode dashboard.","release_4_2.md","product","release_notes","2025-02-14"),
 (16,"Release 3.0 (2022): launched the analytics dashboard and CSV export.","release_3_0.md","product","release_notes","2022-05-30"),
 (17,"Remote work policy: employees may work remotely up to 3 days per week.","hr_policy_2024.pdf","hr","policy","2024-04-01"),
 (18,"Remote work policy: employees must work from the Anantnag office 5 days per week.","hr_policy_2021.pdf","hr","policy","2021-02-15"),
 (19,"Join Veridia Labs, a great place to work with an amazing culture.","careers.html","hr","marketing","2024-05-01"),
 # --- two deliberately awkward docs for the edge cases ---
 (20,"Refunds are sent back to the original card and take 5 business days to arrive.","billing_notes.txt","pricing","faq",None),   # MISSING date
 (21,"Billing security: card details are tokenised by our payment processor and never stored on Veridia servers.","billing_security.pdf","security","policy","2024-03-01"),  # belongs to BOTH pricing and security
]
docs = []
for i,text,src,cat,dtype,date in raw:
    meta = {"doc_id":i,"source":src,"category":cat,"document_type":dtype}
    if date: meta["date"] = date          # doc 20 intentionally has no date key
    docs.append(Document(page_content=text, metadata=meta))
print(len(docs), "documents;", sum('date' not in d.metadata for d in docs), "with missing date")

22 documents; 1 with missing date


## Step 2 - Embed + store vectors *with* their metadata
LangChain's `FAISS.from_documents` embeds each `page_content` and keeps the `metadata` paired with the vector in its docstore. (Local MiniLM embeddings, no OpenAI key needed.)

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store = FAISS.from_documents(docs, emb)
print("vectors in index:", store.index.ntotal)

/tmp/ipykernel_479/2042091131.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

vectors in index: 22


## Step 3 - `filtered_retrieve(query, filters)`
Filters are a dictionary: `category`, `document_type` (exact match) and `min_date` (cutoff, ISO string). Strategy = **filter during search**: we ask FAISS to look at *all* vectors (`fetch_k=len(docs)`), drop anything failing the filter, and return the top-k of what remains. This is why we get k results that obey the constraints, rather than filtering a top-3 that might end up empty.

In [4]:
def make_filter(filters):
    def ok(meta):
        if "category" in filters and meta.get("category") != filters["category"]: return False
        if "document_type" in filters and meta.get("document_type") != filters["document_type"]: return False
        if "min_date" in filters:
            d = meta.get("date")
            if d is None or d < filters["min_date"]: return False   # missing date => excluded (see edge case 1)
        return True
    return ok

def filtered_retrieve(query, filters=None, k=3):
    if not filters:
        return store.similarity_search_with_score(query, k=k)
    return store.similarity_search_with_score(query, k=k, filter=make_filter(filters), fetch_k=len(docs))

def show(results):
    for d,s in results:
        m = d.metadata
        print(f"  [{m['doc_id']:>2}] {s:.3f} | {m['category']:<8} {m['document_type']:<19} {m.get('date','NO DATE'):<10} | {d.page_content[:70]}")

q = "What is the refund policy?"
print("NO FILTER"); show(filtered_retrieve(q))
print("CATEGORY FILTER"); show(filtered_retrieve(q, {"category":"pricing"}))
print("DATE FILTER (>= 2023-01-01)"); show(filtered_retrieve(q, {"min_date":"2023-01-01"}))

NO FILTER
  [ 0] 0.664 | pricing  policy              2021-03-01 | Refund policy (2021): Customers may request a refund within 14 days of
  [ 1] 0.892 | pricing  policy              2024-02-10 | Refund policy (2024): Customers may request a refund within 30 days of
  [20] 0.970 | pricing  faq                 NO DATE    | Refunds are sent back to the original card and take 5 business days to
CATEGORY FILTER
  [ 0] 0.664 | pricing  policy              2021-03-01 | Refund policy (2021): Customers may request a refund within 14 days of
  [ 1] 0.892 | pricing  policy              2024-02-10 | Refund policy (2024): Customers may request a refund within 30 days of
  [20] 0.970 | pricing  faq                 NO DATE    | Refunds are sent back to the original card and take 5 business days to
DATE FILTER (>= 2023-01-01)
  [ 1] 0.892 | pricing  policy              2024-02-10 | Refund policy (2024): Customers may request a refund within 30 days of
  [ 2] 0.974 | pricing  marketing           2024-0

## Step 4 - Five queries, with vs without filters
Ground truth: `answer_ids` = the single document that truly answers the query (current, right type).

Metrics for the top-3:
- **P@1** - is the first result the correct document?
- **Hit@3** - is the correct document anywhere in the top 3?
- **Noise** - how many of the 3 are outdated (before 2023) or marketing fluff.
- **Precision@3** - share of the top 3 that are *useful* (not noise).

In [8]:
tests = [
 ("What is the refund policy?",                 {"category":"pricing","document_type":"policy","min_date":"2023-01-01"},{1}),
 ("How is customer data encrypted?",            {"category":"security","document_type":"technical_reference","min_date":"2023-01-01"},{5}),
 ("When is customer support available?",        {"category":"support","document_type":"faq","min_date":"2023-01-01"},{9}),
 ("What is the API rate limit?",                {"category":"product","document_type":"technical_reference","min_date":"2023-01-01"},{12}),
 ("How many days can employees work remotely?", {"category":"hr","document_type":"policy","min_date":"2023-01-01"},{17}),
]
def is_noise(m): return m["document_type"]=="marketing" or ("date" in m and m["date"]<"2023-01-01")

def score(res, answer_ids):
    ids  = [d.metadata["doc_id"] for d,_ in res]
    noise = sum(is_noise(d.metadata) for d,_ in res)
    return dict(P1=int(bool(ids) and ids[0] in answer_ids), Hit3=int(bool(set(ids)&answer_ids)),
                Noise=noise, P3=round((len(ids)-noise)/max(len(ids),1),2), ids=ids)

import pandas as pd
rows=[]
for q,f,ans in tests:
    a = score(filtered_retrieve(q), ans); b = score(filtered_retrieve(q,f), ans)
    rows.append({"query":q,"no_filter_ids":a["ids"],"filter_ids":b["ids"],
                 "P@1 no":a["P1"],"P@1 filt":b["P1"],"Noise no":a["Noise"],"Noise filt":b["Noise"],
                 "Prec@3 no":a["P3"],"Prec@3 filt":b["P3"]})
df = pd.DataFrame(rows); display(df)
print("Mean P@1       :", df["P@1 no"].mean(), "->", df["P@1 filt"].mean())
print("Mean Noise     :", df["Noise no"].mean(), "->", df["Noise filt"].mean())
print("Mean Precision@3:", round(df["Prec@3 no"].mean(),2), "->", round(df["Prec@3 filt"].mean(),2))

,query,no_filter_ids,filter_ids,P@1 no,P@1 filt,Noise no,Noise filt,Prec@3 no,Prec@3 filt
0,What is the refund policy?,"[0, 1, 20]",[1],0,1,1,0,0.67,1.0
1,How is customer data encrypted?,"[6, 7, 8]",[5],0,1,2,0,0.33,1.0
2,When is customer support available?,"[9, 10, 11]",[9],1,1,2,0,0.33,1.0
3,What is the API rate limit?,"[12, 13, 14]",[12],1,1,2,0,0.33,1.0
4,How many days can employees work remotely?,"[17, 18, 9]",[17],1,1,1,0,0.67,1.0


Mean P@1       : 0.6 -> 1.0
Mean Noise     : 1.6 -> 0.0
Mean Precision@3: 0.47 -> 1.0


## Step 5 - Category filter and date filter on their own
Two separate behaviours, shown separately so each can be verified.

In [6]:
q = "How is customer data encrypted?"
print("Category = security only:"); show(filtered_retrieve(q, {"category":"security"}))
print("\nDate >= 2023-01-01 only:");  show(filtered_retrieve(q, {"min_date":"2023-01-01"}))
print("\nDate >= 2025-01-01 (strict cutoff):"); show(filtered_retrieve(q, {"min_date":"2025-01-01"}))

Category = security only:
  [ 6] 1.135 | security technical_reference 2020-08-05 | Data is encrypted at rest with AES-128 and in transit with TLS 1.2.
  [ 7] 1.186 | security marketing           2024-05-01 | Bank-grade security keeps your data safe. Veridia Labs takes privacy a
  [ 8] 1.251 | security policy              2023-09-01 | Data retention: deleted customer data is permanently purged from backu

Date >= 2023-01-01 only:
  [ 7] 1.186 | security marketing           2024-05-01 | Bank-grade security keeps your data safe. Veridia Labs takes privacy a
  [ 8] 1.251 | security policy              2023-09-01 | Data retention: deleted customer data is permanently purged from backu
  [21] 1.259 | security policy              2024-03-01 | Billing security: card details are tokenised by our payment processor 

Date >= 2025-01-01 (strict cutoff):
  [15] 1.945 | product  release_notes       2025-02-14 | Release 4.2 (2025): added single sign-on via SAML and a dark mode dash


## Step 6 - Edge cases the filter still cannot handle

In [7]:
print("EDGE 1 - missing metadata. 'How long do refunds take to arrive?' (answer is doc 20, which has no date)")
print(" Without filter:"); show(filtered_retrieve("How long do refunds take to arrive?"))
print(" With date filter:"); show(filtered_retrieve("How long do refunds take to arrive?", {"min_date":"2023-01-01"}))

print("\nEDGE 2 - conflicting categories. 'Is my card information stored on your servers?' (doc 21 is billing AND security)")
print(" Filter category=pricing:");  show(filtered_retrieve("Is my card information stored on your servers?", {"category":"pricing"}))
print(" Filter category=security:"); show(filtered_retrieve("Is my card information stored on your servers?", {"category":"security"}))

EDGE 1 - missing metadata. 'How long do refunds take to arrive?' (answer is doc 20, which has no date)
 Without filter:
  [20] 0.532 | pricing  faq                 NO DATE    | Refunds are sent back to the original card and take 5 business days to
  [ 0] 0.702 | pricing  policy              2021-03-01 | Refund policy (2021): Customers may request a refund within 14 days of
  [ 1] 0.955 | pricing  policy              2024-02-10 | Refund policy (2024): Customers may request a refund within 30 days of
 With date filter:
  [ 1] 0.955 | pricing  policy              2024-02-10 | Refund policy (2024): Customers may request a refund within 30 days of
  [ 9] 1.529 | support  faq                 2024-01-08 | Support hours: Monday to Friday, 9am to 6pm IST. Enterprise customers 
  [ 8] 1.555 | security policy              2023-09-01 | Data retention: deleted customer data is permanently purged from backu

EDGE 2 - conflicting categories. 'Is my card information stored on your servers?' (doc 21 is

## Findings
- Mean P@1 rose from 0.6 to 1.0 across the five queries: with filters, the correct current document ranked first every time.
- Mean noise (outdated or marketing chunks in the results) fell from 1.6 to 0.0.
- Mean precision (useful results / results returned) rose from 0.47 to 1.0.
- Without filters, the refund query ranked the 2021 policy first and the encryption query ranked the 2020 AES-128 doc first. Similarity search could not tell old from current.
- A category filter or a date filter alone was not enough; the combination of category, document type and date cutoff removed the noise.
- Caveat: the filters were chosen knowing the right answer's category and type. A real system would need to infer filters from the query.

## Limitations (still not handled)
1. **Missing metadata:** doc 20 has no date. It ranked first for "How long do refunds take to arrive?" without a filter, then disappeared once the date filter was on, with no error.
2. **Conflicting / multi-category docs:** doc 21 (card details are tokenised) is both billing and security. Under `category=pricing` it was invisible and only appeared under `category=security`.